# Phase 7 · step 2 — fit on calibration, rehearse on val

Step 2 of `preregistration/ANALYSIS_PLAN.md` §10, which is **FINAL and binding**.
**CPU only — no GPU hours.** About 5–10 minutes.

| Fitted on the calibration split, per model | Plan |
|---|---|
| **T** — the stage 0+1 temperature (and T for stage 1 alone, D1's comparison) | §4.2 |
| **OOD statistics** — grade means and a Ledoit–Wolf shared covariance on the variant's 5,000 references; the z-scale and **τ_ood**, the 95th percentile | §5.3 |
| **τ_conf** — the 90th percentile of d_conf | §6.1 |
| **r** — the disagreement weight in {0, 0.5, 1.5, 2.5} with the best calibration AUC; ties go to the smaller r | §5.5 |

Then **every table of §§4–8 is rehearsed on val**, labelled as rehearsal. Val is
development data and nothing from it is reported — but it is the last look at what the
locked run will do before anything is committed.

## What it cannot do

It reads no locked data. `analyse.py` refuses the EyePACS test split, APTOS and
Messidor-2 without `--unblind`, and `--unblind` refuses parameters that are not
committed to git. This notebook never passes it.

## Inputs

`verify-dr-internal` (the output of `07_internal_pass.ipynb`) · `verify-dr-manifests`.

| Setting | Value |
|---|---|
| Accelerator | **None** — CPU |
| Persistence | Files only |
| Internet | On |

## 1 · Clone the repo

In [ ]:
import shutil, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)          # always take a clean checkout

# Private repo? Store a GitHub PAT under Add-ons -> Secrets as GH_TOKEN.
# Public repo? Delete the try/except and just clone the plain URL.
url = "https://github.com/kazimab1/DR-New.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    url = url.replace("https://", f"https://{token}@")
    print("cloning with GH_TOKEN")
except Exception:
    print("no GH_TOKEN secret found - cloning anonymously (works if the repo is public)")

!git clone --depth 1 -b claude/charming-faraday-a02cx9 {url} {REPO_DIR} 2>&1 | tail -2

sys.path.insert(0, str(REPO_DIR / "scripts"))
sys.path.insert(0, str(REPO_DIR / "src"))
assert (REPO_DIR / "scripts/build_cache.py").exists(), "clone failed - check the token or branch name"

# Drop verify_dr modules left over from an earlier checkout in this kernel.
# Python caches modules by NAME, not by file, so re-cloning mid-session does
# nothing for an already-imported package: a later cell importing a function
# added upstream still fails with ImportError against the new files on disk.
for _stale in [m for m in list(sys.modules)
               if m == "verify_dr" or m.startswith("verify_dr.")]:
    del sys.modules[_stale]

# Print the commit actually in use. A stale checkout is the single most common
# cause of a confusing failure downstream: the notebook cell is new, the scripts
# on disk are not.
import subprocess
_sha = subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h  %s"],
                      capture_output=True, text=True).stdout.strip()
print("repo ready at", REPO_DIR)
print("checked out:", _sha)

## 2 · Find the internal pass and the manifests

In [ ]:
import json, shlex, subprocess
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

def q(x):
    return shlex.quote(str(x))

def run(cmd):
    """Run a script, streaming its output as it arrives; raise if it fails."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"exit {code}")

def find_internal():
    """The internal pass: a directory holding calibration/ and val/, each with run.json.

    Searched by content, not by a fixed path: a dataset made from notebook output keeps
    its predictions/ folder, one uploaded by hand may not.
    """
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        for hit in sorted(base.rglob("calibration/run.json")):
            root = hit.parent.parent
            if (root / "val" / "run.json").exists():
                return root
    return None

def find_manifest_dir():
    """Phase 2's output: the directory holding dataset_plan.json and the variants."""
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        hits = sorted(base.rglob("dataset_plan.json"))
        if hits:
            return hits[0].parent
    return None

INTERNAL = find_internal()
MANIFEST_DIR = find_manifest_dir()
if INTERNAL is None:
    raise RuntimeError("No internal pass found. Attach verify-dr-internal (07's output).")
if MANIFEST_DIR is None:
    raise RuntimeError("No manifests found. Attach verify-dr-manifests.")
FULL = MANIFEST_DIR / "eyepacs_full.csv"
DDR = MANIFEST_DIR / "eyepacs_ddr_full.csv"

EXPECTED = {"reference_eyepacs_full": 5000, "reference_eyepacs_ddr_full": 5000,
            "calibration": 3512, "val": 7040}
for job, rows in EXPECTED.items():
    images = INTERNAL / job / "images.csv"
    if not images.exists():
        raise RuntimeError(f"{job} is not complete in {INTERNAL}: finish 07 first.")
    with open(images) as fh:
        n = sum(1 for _ in fh) - 1
    print(f"  {job:<28} {n:>6} images" + ("" if n == rows else f"   <-- expected {rows}"))
print("internal pass:", INTERNAL)
print("manifests    :", MANIFEST_DIR)

OUT = WORK / "fitted"
OUT.mkdir(parents=True, exist_ok=True)

## 3 · Fit on the calibration split

Internal labels only: the calibration split's, and the training rows' behind each OOD
reference sample. Writes `fitted/fitted_params.json` and `fitted/ood/<model>.npz`.

In [ ]:
run(" ".join([f"python {q(REPO_DIR / 'scripts/fit_params.py')}",
              f"--internal {q(INTERNAL)}", f"--manifest-full {q(FULL)}",
              f"--manifest-ddr {q(DDR)}", f"--out {q(OUT)}"]))
PARAMS = json.loads((OUT / "fitted_params.json").read_text())

## 4 · What was fitted — and does EM's premise hold?

EM re-estimates a new site's grade mix from its unlabelled images, and it assumes the
average calibrated posterior equals the prior the model was calibrated under. **On the
calibration split nothing has shifted, so EM should hand back the true grade mix.** If
it drifts here, it will drift on APTOS and Messidor-2 whatever their real shift is, and
H2 is in trouble before it starts. `EM drift` is the largest per-grade gap.

In [ ]:
import pandas as pd

# Primary variant first, then by seed -- the JSON itself is sorted alphabetically.
MODELS = sorted(PARAMS["models"], key=lambda n: ("_ddr_" in n, n))
rows = []
for name in MODELS:
    m = PARAMS["models"][name]
    fit, premise = m["calibration_fit"], m["calibration_fit"]["em_premise"]
    rows.append({
        "model": name, "T": round(m["temperature"], 3),
        "T (stage 1 only)": round(m["temperature_stage1_only"], 3),
        "ECE raw": round(fit["ece"]["raw"], 4), "ECE cal": round(fit["ece"]["stage01"], 4),
        "tau_ood": round(m["ood"]["tau_ood"], 3), "tau_conf": round(m["tau_conf"], 4),
        "r": m["r"],
        "EM drift": round(max(abs(a - b) for a, b in
                              zip(premise["em_prior_no_shift"], premise["true_prior"])), 4),
    })
print(pd.DataFrame(rows).to_string(index=False))

print("\nCalibration AUC for each candidate r (the chosen r is the best; ties go smaller):")
for name in MODELS:
    m = PARAMS["models"][name]
    print(f"  {name:<26} " + "  ".join(f"r={k}: {v:.5f}" for k, v in m["r_auc"].items()))

counts = PARAMS["pi_src_counts"]
print(f"\npi_src from {sum(counts)} EyePACS training rows (expect 59,842):",
      [round(v, 4) for v in PARAMS["pi_src"]])
for name in MODELS:
    premise = PARAMS["models"][name]["calibration_fit"]["em_premise"]
    fmt = lambda v: "[" + " ".join(f"{x:.3f}" for x in v) + "]"
    print(f"\n{name}: EM on calibration, where nothing has shifted")
    print(f"  true grade mix  {fmt(premise['true_prior'])}")
    print(f"  mean posterior  {fmt(premise['mean_posterior'])}")
    print(f"  EM's estimate   {fmt(premise['em_prior_no_shift'])}  "
          f"({premise['em_iterations']} iterations)")

## 5 · Rehearse every table on val

Val stands in for every dataset, and `--role external` runs the external-only tables
too (EM, its oracle bound, and the sample-size study), so all the code the unblinding
will run runs here first. Everything printed is labelled **REHEARSAL — not a result**.

In [ ]:
REHEARSAL = OUT / "rehearsal"
run(" ".join([f"python {q(REPO_DIR / 'scripts/analyse.py')} dataset",
              f"--pass-dir {q(INTERNAL / 'val')}", f"--labels {q(FULL)}", "--split val",
              f"--fitted {q(OUT / 'fitted_params.json')}", f"--ood-dir {q(OUT)}",
              "--name val --role external --rehearsal",
              f"--out {q(REHEARSAL / 'val')}"]))

## 6 · Rehearsal verdicts

The §8 claim rule, applied as it will be at the unblinding, with val in every role.
A rehearsal verdict decides nothing.

In [ ]:
run(" ".join([f"python {q(REPO_DIR / 'scripts/analyse.py')} verdicts",
              f"--in-domain {q(REHEARSAL / 'val' / 'results.json')}",
              f"--external {q(REHEARSAL / 'val' / 'results.json')}",
              f"--out {q(REHEARSAL)}"]))

## 7 · The readout

The numbers the next decision rests on, in one place. The evidence table is
descriptive, not a registered outcome: it shows how specific M3 is, which bounds how
informative disagreement can be.

In [ ]:
R = json.loads((REHEARSAL / "val" / "results.json").read_text())
ev = R["evidence"]
print("REHEARSAL on val -- not a result\n")
print("M3's evidence grade against the true grade")
print("  true grade    evidence 0   evidence 1   evidence 2")
for g, row in enumerate(ev["confusion_true_by_evidence"]):
    print(f"  {g:>10}" + "".join(f"{v:>13}" for v in row))
print(f"  M3 QWK {ev['m3_qwk']:.3f}; evidence found in {ev['evidence_any_given_grade0']:.1%} of "
      f"grade-0 images; none found in {ev['evidence0_given_referable']:.1%} of referable ones\n")
arms = ("none", "confidence", "ood", "disagreement", "combined")
for name, r in R["models"].items():
    a = r["arms"]
    blocks = r["signals"]["blocks"]["d_evidence"]
    faith = r["faithfulness"]
    print(f"{name}   accuracy {r['accuracy']:.4f}   r = {r['r']}")
    print("   AUC   " + "   ".join(f"{k} {a[k]['auc']:.4f}" for k in arms))
    print("   d_evidence   " + "   ".join(f"{k}: {v['share']:.0%} of images, M1 right "
                                          f"{v['m1_accuracy']:.0%}" for k, v in blocks.items()))
    share = faith["E1_E2"]["faithful_share"]
    print(f"   faithfulness: {faith['counts']}, faithful share "
          + ("n/a" if share is None else f"{share:.3f}"))

---
## 8 · Save, and what to paste back

1. **Save Version → Quick Save.** Never *Save & Run All*.
2. Publish `/kaggle/working/fitted` as **`verify-dr-fitted`**. The unblinding reads the
   OOD statistics from it, and refuses any whose digest differs from the committed one.
3. Paste back the outputs of **sections 4, 6 and 7**, and the **whole of the cell
   below**: `fitted_params.json`, verbatim.

Nothing is committed from here. Step 3 commits `fitted_params.json` beside the plan and
records the commit in `PREREGISTRATION.md`; the locked pass comes after that commit,
never before.

In [ ]:
print((OUT / "fitted_params.json").read_text())
print("digest:", PARAMS["digest"])